In [0]:
import requests
import pandas as pd 
import pyspark as ps
import time
from pyspark.sql.functions import col, sum as spark_sum, isnan, when, count
from pyspark.sql import SparkSession
from pyspark.sql.types import NullType

In [0]:
indicators = ['NCDMORT3070', 'SDGPM25', 'WHOSIS_000001', 'WHOSIS_000002', 'MDG_0000000029', 'HIV_0000000026', 'NCD_GLUC_04', 'BP_04', 'NCD_BMI_30C', 'M_Est_smk_curr_std', 'WSH_WATER_SAFELY_MANAGED', 'WSH_SANITATION_SAFELY_MANAGED', 'NUTRITION_ANT_HAZ_NE2', 'NUTRITION_ANAEMIA_REPRODUCTIVEAGE_PREV', 'GHED_GGHE-D_pc_US_SHA2011']

In [0]:
base_url = "https://ghoapi.azureedge.net/api/"
dfs = []
erros = []

for code in indicators:
    url = f"{base_url}{code}"
    try:
        response = requests.get(url, timeout=30)
        data = response.json()
        if not data.get('value'):
            print(f"{code} — sem dados, pulando")
            erros.append(code)
            continue

        df_temp = pd.DataFrame(data['value'])
        dfs.append(df_temp)
        print(f"{code} — {len(df_temp)} linhas")
    except Exception as e:
        print(f"{code} — erro: {e}")
        erros.append(code)
    
    time.sleep(2)

df_pandas = pd.concat(dfs, ignore_index=True)
print(f"\nTotal: {df_pandas.shape}")
print(f"Erros: {erros}")

In [0]:
spark = SparkSession.builder.getOrCreate()

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

df_spark = spark.createDataFrame(df_pandas)

for codigo in indicators:
    df_indicador = df_clean.filter(df_clean.IndicatorCode == codigo)
    nome_tabela = codigo.replace("-", "_").replace(".", "_").lower()
    
    df_indicador.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"workspace.bronze.{nome_tabela}")
    
    print(f"✅ workspace.bronze.{nome_tabela} — {df_indicador.count()} linhas")